# 08 — LlamaIndex retriever

**Module notebook — definitions only.**

Scoped to *retrieval only*, inside the RAG agent — nothing else in the project
uses LlamaIndex. It replaces the LangChain/Chroma retriever from
`05_vector_store.ipynb` for the RAG agent's own queries; LangChain still owns
the actual answer generation (prompt + LLM call), reusing
`RAG_SYSTEM_PROMPT_TEMPLATE` and `get_llm()` from `06_rag_engine.ipynb`.

Why swap just the retrieval layer: it's the one piece where LlamaIndex adds
something LangChain's basic retriever doesn't already give you cleanly (its
own chunking/indexing pipeline), without pulling LlamaIndex's own LLM/agent
layer into a project that already has LangGraph doing that job.

Depends on: `00_llm_config.ipynb` (for `get_llm()`, `output_language_instruction()`),
`06_rag_engine.ipynb` (for `RAG_SYSTEM_PROMPT_TEMPLATE`).

In [ ]:
import re
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Multilingual embeddings let Arabic questions retrieve relevant English transcript chunks (and vice versa).
LLAMA_EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
LLAMA_CHUNK_SIZE = 500
LLAMA_CHUNK_OVERLAP = 50
LLAMA_TOP_K = 4
LLAMA_MIN_RELEVANCE_SCORE = 0.35  # below this, treat the question as not covered by the transcript

Settings.embed_model = HuggingFaceEmbedding(model_name=LLAMA_EMBEDDING_MODEL)
Settings.node_parser = SentenceSplitter(chunk_size=LLAMA_CHUNK_SIZE, chunk_overlap=LLAMA_CHUNK_OVERLAP)
Settings.llm = None  # retrieval only - generation stays with LangChain/get_llm()

NOT_COVERED_MESSAGE = {
    "english": "I could not find this information in the meeting transcript.",
    "hinglish": "I could not find this information in the meeting transcript.",
    "arabic": "لم أتمكن من العثور على هذه المعلومة في نص الاجتماع.",
}

GREETING_RESPONSES = {
    "english": "Hello! I am your AI Video Assistant, ready to help you with any questions or details about this recording. How can I help you?",
    "hinglish": "Hello! I am your AI Video Assistant, ready to help you with any questions or details about this recording. How can I help you?",
    "arabic": "أهلاً وسهلاً بك! أنا المساعد الذكي الخاص بالتسجيلات والاجتماعات، جاهز للإجابة عن أي أسئلة أو تفاصيل تخص محتوى هذا التسجيل. كيف يمكنني مساعدتك؟",
}

GREETING_PATTERNS = [
    r"^(?:أ|ا|إ)?هل(?:ا|اً|ا)?(?:\s+(?:وسهلا|بيك|بك|بيكم))?$",
    r"^مرحب(?:ا|اً)?(?:\s+(?:بيك|بك|بيكم|وسهلا))?$",
    r"^سلام(?:\s+(?:عليكم|عليك))?$",
    r"^السلام\s+عليكم(?:\s+ورحم(?:ة|ه)\s+الله(?:\s+وبركات(?:ة|ه))?)?$",
    r"^ازيك(?:\s+(?:عامل\s+ايه|يا\s+بطل|يا\s+غالي|تمام))?$",
    r"^عامل\s+(?:ايه|إيه)$",
    r"^(?:ازيك\s+)?عامل\s+(?:ايه|إيه)$",
    r"^(?:أ|ا)?خبارك|شخبارك|اخباركم|شخباركم$",
    r"^كيف(?:ك|كم|\s+حالك|\s+حالكم|\s+الحال|\s+الأمور)$",
    r"^صباح\s+(?:الخير|النور|الورد)$",
    r"^مساء\s+(?:الخير|النور|الورد)$",
    r"^شلونك|شلونكم$",
    r"^هاي$",
    r"^هلو$",
    r"^(?:شكرا|شكراً|تسلم|تسلملي|تسلموا|يعطيك\s+العافي(?:ة|ه)|يعطيكم\s+العافي(?:ة|ه))$",
    r"^(?:مين\s+انت|من\s+انت|انت\s+مين|بتعمل\s+ايه)$",
    r"^hi(?:ya)?$",
    r"^hello(?:\s+there)?$",
    r"^hey(?:\s+there)?$",
    r"^greetings$",
    r"^good\s+(?:morning|afternoon|evening|day)$",
    r"^how\s+(?:are\s+you(?:\s+doing)?|is\s+it\s+going|do\s+you\s+do)$",
    r"^what(?:'s|\s+is)\s+up$",
    r"^who\s+(?:are\s+you|made\s+you)$",
    r"^what\s+can\s+you\s+do$",
    r"^(?:thanks|thank\s+you)$",
    r"^nice\s+to\s+meet\s+you$",
]

def _normalize_greeting_text(text: str) -> str:
    t = text.strip().lower()
    t = re.sub(r"[ً-ٰٟ]", "", t)
    t = re.sub(r"[\?!,\.،؟!\-_~@#$%^&*()\[\]{}]+", " ", t)
    t = re.sub(r"[إأآا]", "ا", t)
    t = re.sub(r"ة", "ه", t)
    t = re.sub(r"ى", "ي", t)
    return re.sub(r"\s+", " ", t).strip()

_RAW_GREETING_WORDS = {
    "اهلا", "أهلا", "وسهلا", "مرحبا", "سلام", "عليكم", "ورحمة", "ورحمه", "الله", "وبركاته", "وبركته",
    "ازيك", "عامل", "ايه", "إيه", "اخبارك", "أخبارك", "اخباركم", "كيفك", "كيفكم",
    "حالك", "حالكم", "الحال", "صباح", "مساء", "الخير", "النور", "الورد", "هاي",
    "هلو", "شكرا", "تسلم", "تسلملي", "مين", "انت", "إنت", "تمام", "الحمد", "لله",
    "شلونك", "شلونكم", "يا", "بطل", "غالي",
    "hi", "hello", "hey", "how", "are", "you", "good", "morning", "afternoon",
    "evening", "thanks", "thank", "whats", "up", "there", "doing", "fine", "im",
    "i", "am", "who", "what", "can", "do"
}
_GREETING_WORDS = {_normalize_greeting_text(w) for w in _RAW_GREETING_WORDS}

def is_greeting(text: str) -> bool:
    """Return True if the text is a greeting, conversational pleasantry, or small talk."""
    norm = _normalize_greeting_text(text)
    if not norm:
        return False
    for pat in GREETING_PATTERNS:
        if re.match(pat, norm):
            return True
    sub_parts = norm.split(" ")
    if len(sub_parts) <= 7 and all(w in _GREETING_WORDS for w in sub_parts):
        return True
    return False

def detect_question_language(question: str, fallback: str = "english") -> str:
    """Return Arabic for Arabic-script questions, otherwise English."""
    arabic_letters = sum("\u0600" <= char <= "\u06ff" for char in question)
    latin_letters = sum(char.isascii() and char.isalpha() for char in question)
    if arabic_letters > latin_letters:
        return "arabic"
    if latin_letters:
        return "english"
    return fallback if fallback in {"english", "arabic", "hinglish"} else "english"


## Build the index

One in-memory `VectorStoreIndex` per transcript — mirrors `build_vector_store()` in `05_vector_store.ipynb`, just on LlamaIndex's side.

In [ ]:
def build_llama_index_bundle(transcript: str, k: int = LLAMA_TOP_K) -> dict:
    """Index this transcript with LlamaIndex and return a retriever bundle."""
    print("Building LlamaIndex vector index for this transcript.")
    document = Document(text=transcript)
    index = VectorStoreIndex.from_documents([document])
    retriever = index.as_retriever(similarity_top_k=k)
    return {"index": index, "retriever": retriever}


## Retrieve + answer

`retrieve_context` pulls the top-k chunks and also returns the best similarity
score — that score is what lets us catch **edge case: a question with no relevant
chunk in the transcript**. If nothing retrieved clears `LLAMA_MIN_RELEVANCE_SCORE`,
we skip the LLM call entirely and return the "not covered" message directly —
this is a deterministic guardrail, not just a prompt instruction, so it holds even
if the LLM would otherwise be tempted to guess.

`ask_llama_question` then hands the context to the *same* prompt template and LLM
the rest of the project already uses, so answer quality/behavior stays consistent
with `06_rag_engine.ipynb`.

In [ ]:
def retrieve_context(retriever, question: str):
    """Return (context_text, best_score) for the top-k retrieved chunks."""
    nodes = retriever.retrieve(question)
    if not nodes:
        return "", 0.0
    context = "\n\n".join(node.get_content() for node in nodes)
    best_score = max((node.score or 0.0) for node in nodes)
    return context, best_score


def ask_llama_question(index_bundle: dict, question: str, language: str = "english") -> str:
    """Answer in the language used by the question, independent of summary language."""
    answer_language = detect_question_language(question, fallback=language)

    print(f"Question: {question}")
    print(f"RAG answer language: {answer_language}")

    if is_greeting(question):
        answer = GREETING_RESPONSES.get(answer_language, GREETING_RESPONSES["english"])
        print("Greeting detected - returning conversational response.")
        print(f"Answer: {answer}")
        return answer

    context, best_score = retrieve_context(index_bundle["retriever"], question)

    if best_score < LLAMA_MIN_RELEVANCE_SCORE:
        answer = NOT_COVERED_MESSAGE.get(answer_language, NOT_COVERED_MESSAGE["english"])
        print(f"No sufficiently relevant chunk found (best score {best_score:.2f}) - skipping the LLM call.")
        print(f"Answer: {answer}")
        return answer

    llm = get_llm()
    system_prompt = RAG_SYSTEM_PROMPT_TEMPLATE.format(
        lang_instruction=output_language_instruction(answer_language)
    )
    prompt = ChatPromptTemplate.from_messages(
        [("system", system_prompt), ("human", "{question}")]
    )
    chain = prompt | llm | StrOutputParser()

    answer = chain.invoke({"context": context, "question": question})
    print(f"Answer: {answer}")
    return answer
